# Modify Inflow data

This notebook provides tools to load, modify and save inflow input (Simstrat-like `Qin.dat`, `Sin.dat` or `Tin.dat`)

## Content:

- Load and display inflow data
- Modify inflow data
- Save modified inflow data

## Available functions:

- `deep, surf, deep_order, surf_order, units = load_inflow_data(path, file, start='file_start', stop='file_stop', load_units=True)` — Load deep and surface inflows and save their ordering and optionally load units of the inflow
- `save_inflow_data(deep, surf, deep_order, surf_order, path, file, copy=True)` — Save modified inflow data to a new or a existing file
- `deep/surf = check_negative(deep/surf, val_save, var)` — Check for negative values for one inflow
- `change_depth(deep/surf, col_index, depth)` — Change depth of one inflow
- `add_inflow(deep/surf, data_order, col_index=-1, values=0, depth=0)` — Add one inflow
- `remove_inflow(deep/surf, data_order, col_index=-1)` — Remove one inflow
- `change_order(deep/surf, data_order, new_col_index)` — Change order of inflows

## Run the notebook:

To run the Notebook inside the Simstrat build environment go to `Simstrat/tools/ModifyInput` and run:

~~~bash
jupyter nbconvert --to notebook --execute --output Inflow_exc.ipynb Inflow.ipynb
~~~

This will execute `Inflow.ipynb` in a new notebook `Inflow_exc.ipynb`. If you don't need it you can delete the new notebook with:

~~~bash
rm Inflow_exc.ipynb
~~~

or in one line:

~~~bash
jupyter nbconvert --to notebook --execute --output Inflow_exc.ipynb Inflow.ipynb && rm Inflow_exc.ipynb
~~~

---

## Import libraries

In [1]:
# Import standard Python libraries
import os
import numpy as np
import pandas as pd

# Import custom functions
from functions import load_inflow_data, save_inflow_data, check_negative, change_depth, add_inflow, remove_inflow, change_order

## Load inflow data

There are two types of inflows:
- `Deep Inflows`: depth is fixed relative to bottom of lake, saved as `deep`
- `Surface Inflows`: depth is fixed relative to surface of lake, saved as `surf`

Accordingly, two inflow tables are shown. In both tables the first column is the `Datetime` and the following columns are the inflows with their depth and the inflow index (in brackets) in the title.

You can change these parameters:
- `local_path`: the path to the forcing data file
- `file_name`: the name of the forcing data file
  - `Qin.dat` for the water inflow
  - `Sin.dat` for the salinity inflow
  - `Tin.dat` for the temperature inflow
  - other inflows are also possible if they are of the same format
- `start`: from when on to edit forcing data
  - date and time in format `'YYYYMMDDHHMM'` (Year-Month-Day-Hour-Minute)
  - example: `'202510231500'` = October 23, 2025 at 15:00 (3:00 PM)
  - default: `'file_start'` -> start at beginning of file
- `stop`: until when to edit forcing data
  - same format as `start`
  - default: `'file_stop'` -> stop at end of file
- `load_units`: whether to load the units from the file
    - `True` to load the units (default)
    - `False` to not load the units, returns an empty DataFrame to `units` (necessary for `'TestCase_LakeZurich'` inflows)

In [2]:
# Set parameters
local_path = '../../tests/TestCase_LakeZurich'
file_name = 'Qinp.dat'

# Load data
deep, surf, deep_order, surf_order, units = load_inflow_data(
    path = local_path,
    file = file_name,
    start = 'file_start',
    stop = 'file_stop',
    load_units = False
)

# Display units and data
display(units)
print('Deep Inflows')
display(deep.round(4))
print('Surface Inflows')
display(surf.round(4))

# Only non-negative values for water inflow
if file_name == 'Qin.dat':
    positive = True
else:
    positive = False

Loading data from local file: ../../tests/TestCase_LakeZurich\Qinp.dat
✓ Loaded 1 deep inflows with 8 rows from ../../tests/TestCase_LakeZurich\Qinp.dat
✓ Loaded 6 surface inflows with 8 rows from ../../tests/TestCase_LakeZurich\Qinp.dat
✓ Filtered to date range: 8 rows between 2013-11-10 and 2022-12-13


""


Deep Inflows


,Datetime,-10.0 (1)
0,2013-11-10,5.0
1,2014-02-17,5.0
2,2014-02-18,5.0
3,2014-05-28,5.0
4,2014-05-29,5.0
5,2014-09-05,5.0
6,2014-09-05,5.0
7,2022-12-13,5.0


Surface Inflows


,Datetime,-136.0 (1),-5.0 (2),-5.0 (3),-2.0 (4),-2.0 (5),0.0 (6)
0,2013-11-10,0.0,0.0,21.0,21.0,10.0,10.0
1,2014-02-17,0.0,0.0,21.0,21.0,10.0,10.0
2,2014-02-18,0.0,0.0,20.0,20.0,10.0,10.0
3,2014-05-28,0.0,0.0,20.0,20.0,10.0,10.0
4,2014-05-29,0.0,0.0,19.0,19.0,10.0,10.0
5,2014-09-05,0.0,0.0,19.0,19.0,10.0,10.0
6,2014-09-05,0.0,0.0,20.0,20.0,10.0,10.0
7,2022-12-13,0.0,0.0,20.0,20.0,10.0,10.0


## Modify deep inflow data

You can modify the data in the following ways:
- `Add a constant`
- `Multiply by a constant`
- `Set to a constant`
- `Add a linear trend`
- `Change depth`
- `Add an inflow`
- `Remove an inflow`
- `Change order of inflows`

Be aware that:
- Water deep inflows (`Qin.dat` files) cannot be negative
    - this is controlled in this code
- If `ModelConfig.InflowMode` in Simstrat is set to 1 then:
    - all deep inflow values must be given for a range of depths on a per-meter basis (Q/h), as they will be integrated over depth by the model
    - more info here: https://github.com/Eawag-AppliedSystemAnalysis/Simstrat/blob/master/doc/SIMSTRAT_V304_UserManual.pdf, Section 5.2
    - this must be assured manually

### Add constant

You can change these parameters:
- `index`: the index of the deep inflow to modify
    - any integer with an existing deep inflow
- `constant`: the constant to add
    - any real number (can also be negative and/or a fraction)
    - water deep inflows cannot become negative

In [3]:
# Set parameters
index = 1
constant = -2

# Modify and display data
val_save = deep.iloc[:, index].copy()
deep.iloc[:, index] += float(constant)
if positive:
    deep.iloc[:, index] = check_negative(
        deep.iloc[:, index], val_save, str(index))
else:
    print(f"✓ Inflow ({index}) modified")
display(deep.round(4))

✓ Inflow (1) modified


,Datetime,-10.0 (1)
0,2013-11-10,3.0
1,2014-02-17,3.0
2,2014-02-18,3.0
3,2014-05-28,3.0
4,2014-05-29,3.0
5,2014-09-05,3.0
6,2014-09-05,3.0
7,2022-12-13,3.0


### Multiply by constant

You can change these parameters:
- `index`: the index of the deep inflow to modify
    - any integer with an existing deep inflow
- `constant`: the constant to multiply by
    - any real number (can also be negative and/or a fraction)
    - water deep inflows cannot become negative

In [4]:
# Set parameters
index = 1
constant = 1/5

# Modify and display data
val_save = deep.iloc[:, index].copy()
deep.iloc[:, index] *= float(constant)
if positive:
    deep.iloc[:, index] = check_negative(
        deep.iloc[:, index], val_save, str(index))
else:
    print(f"✓ Inflow ({index}) modified")
display(deep.round(4))

✓ Inflow (1) modified


,Datetime,-10.0 (1)
0,2013-11-10,0.6
1,2014-02-17,0.6
2,2014-02-18,0.6
3,2014-05-28,0.6
4,2014-05-29,0.6
5,2014-09-05,0.6
6,2014-09-05,0.6
7,2022-12-13,0.6


### Set to constant

You can change these parameters:
- `index`: the index of the deep inflow to modify
    - any integer with an existing deep inflow
- `constant`: the constant value of the deep inflow
    - any real number (can also be negative and/or a fraction)
    - water deep inflows cannot become negative

In [5]:
# Set parameters
index = 1
constant = 1

# Modify and display data
val_save = deep.iloc[:, index].copy()
deep.iloc[:, index] = np.full((len(deep)), float(constant))
if positive:
    deep.iloc[:, index] = check_negative(
        deep.iloc[:, index], val_save, str(index))
else:
    print(f"✓ Inflow ({index}) modified")
display(deep.round(4))

✓ Inflow (1) modified


,Datetime,-10.0 (1)
0,2013-11-10,1.0
1,2014-02-17,1.0
2,2014-02-18,1.0
3,2014-05-28,1.0
4,2014-05-29,1.0
5,2014-09-05,1.0
6,2014-09-05,1.0
7,2022-12-13,1.0


### Add linear trend

- `index`: the index of the deep inflow to modify
    - any integer with an existing deep inflow
- `step`: the time of change
    - any positive integer
    - usually `1` corresponds to one hour
    - accordingly `24` corresponds to one day
    - `8760` corresponds to one year
    - For `'TestCase_LakeZurich'` deep inflows `1` is of varying length
- `slope`: the amount of change per time of change
    - any real number (can also be negative and/or a fraction)
    - water deep inflows cannot become negative
    - deep inflow `(index)` increases by `slope` every `step` hours
- `offset`: the offset to the trend
    - any real number (can also be negative and/or a fraction)
    - water deep inflows cannot become negative
    - default is `0`

In [6]:
# Set parameters
index = 1
step = 1
slope = 1/7
offset = 0

# Modify and display data
rate = float(slope / step)
val_save = deep.iloc[:, index].copy()
deep.iloc[:, index] = (deep.iloc[:, index] +
    np.linspace(0, rate * (len(deep) - 1), len(deep)) +
    float(offset))
if positive:
    deep.iloc[:, index] = check_negative(
        deep.iloc[:, index], val_save, str(index))
else:
    print(f"✓ Inflow ({index}) modified")
display(deep.round(4))

✓ Inflow (1) modified


,Datetime,-10.0 (1)
0,2013-11-10,1.0000
1,2014-02-17,1.1429
2,2014-02-18,1.2857
3,2014-05-28,1.4286
4,2014-05-29,1.5714
5,2014-09-05,1.7143
6,2014-09-05,1.8571
7,2022-12-13,2.0000


### Change depth

You can change these parameters:
- `index`: the index of the deep inflow to modify
    - any integer with an existing deep inflow
- `depth`: the new depth
    - any real number (can also be negative and/or a fraction)
    - water deep inflows cannot become negative

In [7]:
# Set parameters
index = 1
depth = -2

# Modify and display data
deep = change_depth(deep, index, depth)
display(deep.round(4))

✓ Modified depth of inflow (1)


,Datetime,-2.0 (1)
0,2013-11-10,1.0000
1,2014-02-17,1.1429
2,2014-02-18,1.2857
3,2014-05-28,1.4286
4,2014-05-29,1.5714
5,2014-09-05,1.7143
6,2014-09-05,1.8571
7,2022-12-13,2.0000


### Add inflow

You can change these parameters:
- `index`: the index of the new deep inflow
    - any positive integer not larger than the amount of existing deep inflows plus one, or `-1`
    - default is `-1` to append as the last deep inflow
- `values`: the values of the new deep inflow, can be:
    - any real number (can also be negative and/or a fraction): to set a constant value in time
    - 1D array of real numbers (can also be negative and/or a fraction) of the same length as the other deep inflows
    - water deep inflows cannot become negative
    - default is `0`, constant in time
- `depth`: the depth of the new deep inflow
    - any real number (can also be negative and/or a fraction)
    - default is `0`

In [8]:
# Set parameters
index = 1
values = 5
depth = 0

# Modify and display data
deep, deep_order = add_inflow(
    deep, deep_order, index, values, depth, positive)
display(deep.round(4))

✓ Added inflow at (1)


,Datetime,0.0 (1),-2.0 (2)
0,2013-11-10,5.0,1.0000
1,2014-02-17,5.0,1.1429
2,2014-02-18,5.0,1.2857
3,2014-05-28,5.0,1.4286
4,2014-05-29,5.0,1.5714
5,2014-09-05,5.0,1.7143
6,2014-09-05,5.0,1.8571
7,2022-12-13,5.0,2.0000


### Remove inflow

You can change these parameters:
- `index`: the index of the deep inflow to remove
    - any integer with an existing deep inflow, or `-1`
    - default is `-1` to remove the last deep inflow

In [9]:
# Set parameter
index = 1

# Modify and display data
deep, deep_order = remove_inflow(
    deep, deep_order, index)
display(deep.round(4))

✓ Removed inflow (1)


,Datetime,-2.0 (1)
0,2013-11-10,1.0000
1,2014-02-17,1.1429
2,2014-02-18,1.2857
3,2014-05-28,1.4286
4,2014-05-29,1.5714
5,2014-09-05,1.7143
6,2014-09-05,1.8571
7,2022-12-13,2.0000


### Change order

You can change these parameters:
- `order`: the new positions of the deep inflows
    - tuple with as many elements as existing deep inflows
    - example: `[2, 1]`: deep inflows (1) and (2) change position

In [10]:
# Set parameter
order = [1]

# Modify and display data
deep, deep_order = change_order(
    deep, deep_order, order)
display(deep.round(4))

✓ Changed inflow order


,Datetime,-2.0 (1)
0,2013-11-10,1.0000
1,2014-02-17,1.1429
2,2014-02-18,1.2857
3,2014-05-28,1.4286
4,2014-05-29,1.5714
5,2014-09-05,1.7143
6,2014-09-05,1.8571
7,2022-12-13,2.0000


# Step 4
## Modify surface inflow data

You can modify the data in the following ways:
- `Add a constant`
- `Multiply by a constant`
- `Set to a constant`
- `Add a linear trend`
- `Change depth`
- `Add an inflow`
- `Remove an inflow`
- `Change order of inflows`

Be aware that:
- Water surface inflows (`Qin.dat` files) cannot be negative
    - this is controlled in this code
- All surface inflow values must be given for a range of depths on a per-meter basis (Q/h), as they will be integrated over depth by the model
    - more info here: https://github.com/Eawag-AppliedSystemAnalysis/Simstrat/blob/master/doc/SIMSTRAT_V304_UserManual.pdf, Section 5.2
    - this must be assured manually

### Add constant

You can change these parameters:
- `index`: the index of the surface inflow to modify
    - any integer with an existing surface inflow
- `constant`: the constant to add
    - any real number (can also be negative and/or a fraction)
    - water surface inflows cannot become negative

In [11]:
# Set parameters
index = 5
constant = -5

# Modify and display data
val_save = surf.iloc[:, index].copy()
surf.iloc[:, index] += float(constant)
if positive:
    surf.iloc[:, index] = check_negative(
        surf.iloc[:, index], val_save, str(index))
else:
    print(f"✓ Inflow ({index}) modified")
display(surf.round(4))

✓ Inflow (5) modified


,Datetime,-136.0 (1),-5.0 (2),-5.0 (3),-2.0 (4),-2.0 (5),0.0 (6)
0,2013-11-10,0.0,0.0,21.0,21.0,5.0,10.0
1,2014-02-17,0.0,0.0,21.0,21.0,5.0,10.0
2,2014-02-18,0.0,0.0,20.0,20.0,5.0,10.0
3,2014-05-28,0.0,0.0,20.0,20.0,5.0,10.0
4,2014-05-29,0.0,0.0,19.0,19.0,5.0,10.0
5,2014-09-05,0.0,0.0,19.0,19.0,5.0,10.0
6,2014-09-05,0.0,0.0,20.0,20.0,5.0,10.0
7,2022-12-13,0.0,0.0,20.0,20.0,5.0,10.0


### Multiply by constant

 You can change these parameters:
- `index`: the index of the surface inflow to modify
    - any integer with an existing surface inflow
- `constant`: the constant to multiply by
    - any real number (can also be negative and/or a fraction)
    - water surface inflows cannot become negative

In [12]:
# Set paramters
index = 6
constant = 1/2

# Modify and display data
val_save = surf.iloc[:, index].copy()
surf.iloc[:, index] *= float(constant)
if positive:
    surf.iloc[:, index] = check_negative(
        surf.iloc[:, index], val_save, str(index))
else:
    print(f"✓ Inflow ({index}) modified")
display(surf.round(4))

✓ Inflow (6) modified


,Datetime,-136.0 (1),-5.0 (2),-5.0 (3),-2.0 (4),-2.0 (5),0.0 (6)
0,2013-11-10,0.0,0.0,21.0,21.0,5.0,5.0
1,2014-02-17,0.0,0.0,21.0,21.0,5.0,5.0
2,2014-02-18,0.0,0.0,20.0,20.0,5.0,5.0
3,2014-05-28,0.0,0.0,20.0,20.0,5.0,5.0
4,2014-05-29,0.0,0.0,19.0,19.0,5.0,5.0
5,2014-09-05,0.0,0.0,19.0,19.0,5.0,5.0
6,2014-09-05,0.0,0.0,20.0,20.0,5.0,5.0
7,2022-12-13,0.0,0.0,20.0,20.0,5.0,5.0


### Set to constant

You can change these parameters:
- `index`: the index of the surface inflow to modify
    - any integer with an existing surface inflow
- `constant`: the constant value of the surface inflow
    - any real number (can also be negative and/or a fraction)
    - water surface inflows cannot become negative

In [13]:
# Set parameters
index = 2
constant = 3

# Modify and display data
val_save = surf.iloc[:, index].copy()
surf.iloc[:, index] = np.full((len(surf)), float(constant))
if positive:
    surf.iloc[:, index] = check_negative(
        surf.iloc[:, index], val_save, str(index))
else:
    print(f"✓ Inflow ({index}) modified")
display(surf.round(4))

✓ Inflow (2) modified


,Datetime,-136.0 (1),-5.0 (2),-5.0 (3),-2.0 (4),-2.0 (5),0.0 (6)
0,2013-11-10,0.0,3.0,21.0,21.0,5.0,5.0
1,2014-02-17,0.0,3.0,21.0,21.0,5.0,5.0
2,2014-02-18,0.0,3.0,20.0,20.0,5.0,5.0
3,2014-05-28,0.0,3.0,20.0,20.0,5.0,5.0
4,2014-05-29,0.0,3.0,19.0,19.0,5.0,5.0
5,2014-09-05,0.0,3.0,19.0,19.0,5.0,5.0
6,2014-09-05,0.0,3.0,20.0,20.0,5.0,5.0
7,2022-12-13,0.0,3.0,20.0,20.0,5.0,5.0


### Add linear trend

You can change these parameters:
- `index`: the index of the surface inflow to modify
    - any integer with an existing surface inflow
- `step`: the time of change
    - any positive integer
    - usually `1` corresponds to one hour
    - accordingly `24` corresponds to one day
    - `8760` corresponds to one year
    - For `'TestCase_LakeZurich'` surface inflows `1` is of varying length
- `slope`: the amount of change per time of change
    - any real number (can also be negative and/or a fraction)
    - water surface inflows cannot become negative
    - inflow `(index)` increases by `slope` every `step` hours
- `offset`: the offset to the trend
    - any real number (can also be negative and/or a fraction)
    - water surface inflows cannot become negative
    - default is `0`

In [14]:
# Set parameters
index = 2
step = 1
slope = 2/7
offset = -2

# Modify and display data
rate = float(slope / step)
val_save = surf.iloc[:, index].copy()
surf.iloc[:, index] = (surf.iloc[:, index] +
    np.linspace(0, rate * (len(surf) - 1), len(surf)) +
    float(offset))
if positive:
    surf.iloc[:, index] = check_negative(
        surf.iloc[:, index], val_save, str(index))
else:
    print(f"✓ Inflow ({index}) modified")
display(surf.round(4))

✓ Inflow (2) modified


,Datetime,-136.0 (1),-5.0 (2),-5.0 (3),-2.0 (4),-2.0 (5),0.0 (6)
0,2013-11-10,0.0,1.0000,21.0,21.0,5.0,5.0
1,2014-02-17,0.0,1.2857,21.0,21.0,5.0,5.0
2,2014-02-18,0.0,1.5714,20.0,20.0,5.0,5.0
3,2014-05-28,0.0,1.8571,20.0,20.0,5.0,5.0
4,2014-05-29,0.0,2.1429,19.0,19.0,5.0,5.0
5,2014-09-05,0.0,2.4286,19.0,19.0,5.0,5.0
6,2014-09-05,0.0,2.7143,20.0,20.0,5.0,5.0
7,2022-12-13,0.0,3.0000,20.0,20.0,5.0,5.0


### Change depth

You can change these parameters:
- `index`: the index of the surface inflow to modify
    - any integer with an existing surface inflow
- `depth`: the new depth
    - any real number (can also be negative and/or a fraction)

In [15]:
# Set parameters
index = 4
depth = -1

# Modify data
surf = change_depth(surf, index, depth)

# Set parameters
index = 5
depth = -1

# Modify and display data
surf = change_depth(surf, index, depth)
display(surf.round(4))

✓ Modified depth of inflow (4)
✓ Modified depth of inflow (5)


,Datetime,-136.0 (1),-5.0 (2),-5.0 (3),-1.0 (4),-1.0 (5),0.0 (6)
0,2013-11-10,0.0,1.0000,21.0,21.0,5.0,5.0
1,2014-02-17,0.0,1.2857,21.0,21.0,5.0,5.0
2,2014-02-18,0.0,1.5714,20.0,20.0,5.0,5.0
3,2014-05-28,0.0,1.8571,20.0,20.0,5.0,5.0
4,2014-05-29,0.0,2.1429,19.0,19.0,5.0,5.0
5,2014-09-05,0.0,2.4286,19.0,19.0,5.0,5.0
6,2014-09-05,0.0,2.7143,20.0,20.0,5.0,5.0
7,2022-12-13,0.0,3.0000,20.0,20.0,5.0,5.0


### Add inflow

You can change these parameters:
- `index`: the index of the new surface inflow
    - any positive integer not larger than the amount of existing deep inflows plus one, or `-1`
    - default is `-1` to append as the last inflow
- `values`: the values of the new surface inflow, can be:
    - any real number (can also be negative and/or a fraction): to set a constant value in time
    - 1D array of real numbers (can also be negative and/or a fraction) of the same length as the other surface inflows
    - water surface inflows cannot become negative
    - default is `0`, constant in time
- `depth`: the depth of the new surface inflow
    - any real number (can also be negative and/or a fraction)
    - default is `0`

In [16]:
# Set parameter
index = 1
values = 0
depth = -5

# Modify and display data
surf, surf_order = add_inflow(
    surf, surf_order, index, values, depth, positive)
display(surf.round(4))

✓ Added inflow at (1)


,Datetime,-5.0 (1),-136.0 (2),-5.0 (3),-5.0 (4),-1.0 (5),-1.0 (6),0.0 (7)
0,2013-11-10,0.0,0.0,1.0000,21.0,21.0,5.0,5.0
1,2014-02-17,0.0,0.0,1.2857,21.0,21.0,5.0,5.0
2,2014-02-18,0.0,0.0,1.5714,20.0,20.0,5.0,5.0
3,2014-05-28,0.0,0.0,1.8571,20.0,20.0,5.0,5.0
4,2014-05-29,0.0,0.0,2.1429,19.0,19.0,5.0,5.0
5,2014-09-05,0.0,0.0,2.4286,19.0,19.0,5.0,5.0
6,2014-09-05,0.0,0.0,2.7143,20.0,20.0,5.0,5.0
7,2022-12-13,0.0,0.0,3.0000,20.0,20.0,5.0,5.0


### Remove inflow

You can change these parameters:
- `index`: the index of the surface inflow to remove
    - any integer with an existing surface inflow, or `-1`
    - default is `-1` to remove the last surface inflow

In [17]:
# Set parameter
index = 3

# Modify and display data
surf, surf_order = remove_inflow(
    surf, surf_order, index)
display(surf.round(4))

✓ Removed inflow (3)


,Datetime,-5.0 (1),-136.0 (2),-5.0 (3),-1.0 (4),-1.0 (5),0.0 (6)
0,2013-11-10,0.0,0.0,21.0,21.0,5.0,5.0
1,2014-02-17,0.0,0.0,21.0,21.0,5.0,5.0
2,2014-02-18,0.0,0.0,20.0,20.0,5.0,5.0
3,2014-05-28,0.0,0.0,20.0,20.0,5.0,5.0
4,2014-05-29,0.0,0.0,19.0,19.0,5.0,5.0
5,2014-09-05,0.0,0.0,19.0,19.0,5.0,5.0
6,2014-09-05,0.0,0.0,20.0,20.0,5.0,5.0
7,2022-12-13,0.0,0.0,20.0,20.0,5.0,5.0


### Change order

You can change these parameters:
- `order`: the new positions of the surface inflows
    - tuple with as many elements as existing surface inflows
    - example: `[2, 1, 3, 4, 5, 6]`: surface inflows (1) and (2) change position

In [18]:
# Set parameters
order = [2, 1, 3, 4, 5, 6]

# Modify and display data
surf, surf_order = change_order(
    surf, surf_order, order)
display(surf.round(4))

✓ Changed inflow order


,Datetime,-136.0 (1),-5.0 (2),-5.0 (3),-1.0 (4),-1.0 (5),0.0 (6)
0,2013-11-10,0.0,0.0,21.0,21.0,5.0,5.0
1,2014-02-17,0.0,0.0,21.0,21.0,5.0,5.0
2,2014-02-18,0.0,0.0,20.0,20.0,5.0,5.0
3,2014-05-28,0.0,0.0,20.0,20.0,5.0,5.0
4,2014-05-29,0.0,0.0,19.0,19.0,5.0,5.0
5,2014-09-05,0.0,0.0,19.0,19.0,5.0,5.0
6,2014-09-05,0.0,0.0,20.0,20.0,5.0,5.0
7,2022-12-13,0.0,0.0,20.0,20.0,5.0,5.0


# Step 5
## Save modified data

You can change this parameter:
- `copy`: Wheter to create a new inflow file (with _modified appended to the file_name) or to overwrite the original
    - `True` to create a new file (default)
    - `False` to overwrite the original

In [19]:
# Save data
save_inflow_data(
    deep,
    surf,
    deep_order,
    surf_order,
    path = local_path,
    file = file_name,
    copy = True
)

Writing modified data into: ../../tests/TestCase_LakeZurich\Qinp_modified.dat
✓ Modified 8 rows
